<a href="https://colab.research.google.com/github/azraayasminn/Ecommerce_AI/blob/main/Laporan_%26_Dashboard_Analisis_Data_Ecommerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Anggota Kelompok 5:

1. Azra Yasmin Meisyifa
2. Clarisya Aura Islami Purwasaputri
3. Gian Aditya Mahardika
4. M Viorel Sadra Asmara

## Skenario
Mengevaluasi performa penjualan periode terakhir untuk menyusun strategi promosi periode berikutnya.

## Alur Pengerjaan
1. Memuat dan membersihkan data (cek data kosong dan duplikat).
2. Menghitung statistik deskriptif kunci.
3. Membuat visualisasi sesuai pertanyaan bisnis.
4. Menyusun dashboard gabungan (subplot).
5. Menyusun laporan rekomendasi bisnis.

In [ ]:
# ============================================================
# 0. IMPOR LIBRARY DAN FUNGSI BANTU
# ============================================================
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display

warnings.filterwarnings("ignore")

LEBAR = 65


def judul(teks):
    """Cetak judul bagian dengan garis pemisah."""
    print("\n" + "=" * LEBAR)
    print(teks)
    print("=" * LEBAR)


def batas_iqr(seri):
    """Hitung Q1, Q3, IQR, serta batas bawah dan atas outlier."""
    q1, q3 = seri.quantile(0.25), seri.quantile(0.75)
    iqr = q3 - q1
    return q1, q3, iqr, q1 - 1.5 * iqr, q3 + 1.5 * iqr


def cari_outlier(df, kolom):
    """Kembalikan baris yang nilainya di luar batas IQR."""
    _, _, _, bawah, atas = batas_iqr(df[kolom])
    return df[(df[kolom] < bawah) | (df[kolom] > atas)]


KOLOM_NUMERIK = ["Age", "Quantity", "Unit_Price", "Discount",
                 "Rating", "Delivery_Days", "Sales", "Profit"]

KOLOM_KATEGORI = ["Gender", "City", "Category", "Sub_Category",
                  "Product", "Payment_Method", "Shipping_Mode",
                  "Returned", "Customer_Type"]

KOLOM_WAJIB = ["Sales", "Profit", "Category", "City",
               "Rating", "Order_Date", "Quantity"]

judul("ANALISIS DATA PENJUALAN E-COMMERCE")

In [ ]:
# ============================================================
# 1. MEMUAT DATA
# ============================================================
judul("[1] UPLOAD DATASET CSV")

uploaded = files.upload()
file_name = next((n for n in uploaded if n.lower().endswith(".csv")), None)

if file_name is None:
    raise ValueError("File CSV tidak ditemukan. Silakan upload file CSV.")

df = pd.read_csv(file_name)

print("\nData berhasil dimuat!")
print(f"Nama file    : {file_name}")
print(f"Jumlah baris : {df.shape[0]}")
print(f"Jumlah kolom : {df.shape[1]}")

print("\nLima data pertama:")
display(df.head())

print("\nInformasi dataset:")
df.info()


[1] UPLOAD DATASET CSV


## 2. Eksplorasi dan Pembersihan Data
Pada tahap ini dilakukan pengecekan data kosong dan duplikat, serta ditampilkan ringkasan statistik awal (`describe`). Langkah pembersihan:
- Menghapus baris duplikat (20 baris).
- Merapikan spasi pada teks dan menyeragamkan huruf kategori (mis. "male" dan "MALE" menjadi "Male").
- Mengubah tipe data tanggal dan angka.
- Mengisi data numerik kosong dengan **median** (tahan terhadap nilai ekstrem).
- Mengisi data teks kosong dengan **modus** (atau "Unknown").

Setelah pembersihan tersisa 2.500 baris tanpa data kosong. Data bersih disimpan sebagai `ecommerce_sales_cleaned.csv`.

In [ ]:
# --- Kondisi awal ---
jumlah_baris_awal = len(df)
duplikat_awal = int(df.duplicated().sum())
missing_awal = int(df.isna().sum().sum())

print("\nData kosong sebelum dibersihkan:")
display(
    df.isna().sum()
      .loc[lambda x: x > 0]
      .sort_values(ascending=False)
      .rename("Jumlah Data Kosong")
      .to_frame()
)
print("Total data kosong :", missing_awal)
print("Baris duplikat    :", duplikat_awal)

print("\nStatistik deskriptif awal:")
display(df.describe(include="all").T)

# --- Pembersihan ---
# 1. Hapus duplikat identik
df = df.drop_duplicates().copy()

# 2. Rapikan spasi pada teks
kolom_teks = df.select_dtypes(include=["object", "string"]).columns
for col in kolom_teks:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

# 3. Ubah tipe data
if "Order_Date" in df.columns:
    df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

for col in KOLOM_NUMERIK:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 4. Seragamkan huruf kategori (mis. "male" dan "MALE" menjadi "Male")
for col in KOLOM_KATEGORI:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: x.title() if isinstance(x, str) else x)

# 5. Isi numerik kosong dengan median (tahan terhadap nilai ekstrem)
for col in KOLOM_NUMERIK:
    if col in df.columns and df[col].isna().any():
        median_value = df[col].median()
        if pd.notna(median_value):
            df[col] = df[col].fillna(median_value)

# 6. Isi teks kosong dengan modus, atau "Unknown" jika tidak ada modus
for col in df.select_dtypes(include=["object", "string"]).columns:
    if df[col].isna().any():
        modus = df[col].mode(dropna=True)
        df[col] = df[col].fillna(modus.iloc[0] if not modus.empty else "Unknown")

# 7. Tanggal tidak valid: isi dengan tanggal valid yang paling sering muncul
if "Order_Date" in df.columns and df["Order_Date"].isna().any():
    modus_tanggal = df["Order_Date"].mode()
    if not modus_tanggal.empty:
        df["Order_Date"] = df["Order_Date"].fillna(modus_tanggal.iloc[0])

# --- Pemeriksaan setelah pembersihan ---
print("\nRingkasan pembersihan:")
print("Baris awal                   :", jumlah_baris_awal)
print("Duplikat yang dihapus        :", duplikat_awal)
print("Baris setelah hapus duplikat :", len(df))
print("Data kosong tersisa          :", int(df.isna().sum().sum()))

print("\nData kosong per kolom setelah dibersihkan:")
display(df.isna().sum().rename("Data Kosong").to_frame())

print("\nLima data setelah pembersihan:")
display(df.head())

print("\nStatistik deskriptif setelah pembersihan:")
display(df.describe().T)

df.to_csv("ecommerce_sales_cleaned.csv", index=False)
print("\nFile data bersih berhasil dibuat: ecommerce_sales_cleaned.csv")

In [ ]:
# ============================================================
# 3. STATISTIK DESKRIPTIF DAN KPI BISNIS
# ============================================================
judul("[3] STATISTIK DESKRIPTIF DAN KPI BISNIS")

kolom_hilang = [c for c in KOLOM_WAJIB if c not in df.columns]
if kolom_hilang:
    raise ValueError(f"Kolom penting tidak ditemukan: {kolom_hilang}")

# --- KPI utama ---
total_penjualan = df["Sales"].sum()
rata_transaksi = df["Sales"].mean()
median_penjualan = df["Sales"].median()
std_penjualan = df["Sales"].std()
q1 = df["Sales"].quantile(0.25)
q3 = df["Sales"].quantile(0.75)
total_profit = df["Profit"].sum()
rata_rating = df["Rating"].mean()
rata_quantity = df["Quantity"].mean()

# --- Peringkat kategori dan kota ---
kategori_penjualan = df.groupby("Category")["Sales"].sum()
kategori_profit = df.groupby("Category")["Profit"].sum()
kota_penjualan = df.groupby("City")["Sales"].sum()

kategori_terlaris = kategori_penjualan.idxmax()
kategori_profit_tertinggi = kategori_profit.idxmax()
wilayah_tertinggi = kota_penjualan.idxmax()
kota_terendah = kota_penjualan.idxmin()

jumlah_transaksi = df["Order_ID"].nunique() if "Order_ID" in df.columns else len(df)

print(f"Jumlah transaksi unik        : {jumlah_transaksi:,}")
print(f"Total penjualan              : {total_penjualan:,.2f} (satuan dataset)")
print(f"Rata-rata nilai transaksi    : {rata_transaksi:,.2f}")
print(f"Median nilai penjualan       : {median_penjualan:,.2f}")
print(f"Standar deviasi penjualan    : {std_penjualan:,.2f}")
print(f"Kuartil pertama (Q1)         : {q1:,.2f}")
print(f"Kuartil ketiga (Q3)          : {q3:,.2f}")
print(f"Total profit                 : {total_profit:,.2f} (satuan dataset)")
print(f"Rata-rata rating             : {rata_rating:.2f} / 5")
print(f"Rata-rata jumlah produk      : {rata_quantity:.2f}")
print(f"Kategori penjualan tertinggi : {kategori_terlaris}")
print(f"Kota penjualan tertinggi     : {wilayah_tertinggi}")
print(f"Kategori profit tertinggi    : {kategori_profit_tertinggi}")

# --- Tabel statistik utama ---
kolom_stat = ["Sales", "Profit", "Quantity", "Rating"]
statistik = df[kolom_stat].agg(["count", "mean", "median", "std", "min", "max"]).T
statistik["Q1"] = df[kolom_stat].quantile(0.25)
statistik["Q3"] = df[kolom_stat].quantile(0.75)

print("\nTabel statistik utama:")
display(statistik.round(2))

In [ ]:
# ============================================================
# 4. VISUALISASI 1: BAR CHART (Penjualan per Kategori)
# ============================================================
judul("[4] VISUALISASI DAN INSIGHT")

plt.figure(figsize=(10, 6))
kategori_penjualan.sort_values().plot(kind="barh", edgecolor="black")
plt.title("Total Penjualan Berdasarkan Kategori Produk")
plt.xlabel("Total Penjualan (satuan dataset)")
plt.ylabel("Kategori Produk")
plt.tight_layout()
plt.show()

kategori_terendah = kategori_penjualan.idxmin()

print("\nINSIGHT 1 - BAR CHART")
print(f"Kategori {kategori_terlaris} memiliki total penjualan tertinggi "
      f"sebesar {kategori_penjualan.max():,.2f}.")
print(f"Kategori {kategori_terendah} memiliki total penjualan terendah "
      f"sebesar {kategori_penjualan.min():,.2f}. Prioritas stok dan promosi "
      "perlu dievaluasi berdasarkan penjualan sekaligus profit setiap kategori.")

In [ ]:
# ============================================================
# 5. VISUALISASI 2: LINE CHART (Tren Penjualan Bulanan)
# ============================================================
df["Month"] = df["Order_Date"].dt.to_period("M").astype(str)
penjualan_bulanan = df.groupby("Month")["Sales"].sum().sort_index()

plt.figure(figsize=(12, 6))
plt.plot(penjualan_bulanan.index, penjualan_bulanan.values, marker="o", linewidth=2)
plt.title("Tren Total Penjualan Bulanan")
plt.xlabel("Bulan")
plt.ylabel("Total Penjualan (satuan dataset)")
plt.xticks(rotation=45, ha="right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

bulan_puncak = penjualan_bulanan.idxmax()
bulan_terendah = penjualan_bulanan.idxmin()

print("\nINSIGHT 2 - LINE CHART")
print(f"Penjualan bulanan tertinggi terjadi pada {bulan_puncak}, "
      f"sebesar {penjualan_bulanan.max():,.2f}.")
print(f"Penjualan bulanan terendah terjadi pada {bulan_terendah}, "
      f"sebesar {penjualan_bulanan.min():,.2f}. Pola ini dapat digunakan "
      "untuk merencanakan promosi dan stok, tetapi penyebab perubahan "
      "perlu diperiksa lebih lanjut.")

In [ ]:
# ============================================================
# 6. VISUALISASI 3: SCATTER PLOT (Quantity vs Sales)
# ============================================================
plt.figure(figsize=(9, 6))
plt.scatter(df["Quantity"], df["Sales"], alpha=0.55)
plt.title("Hubungan Jumlah Produk dan Nilai Penjualan")
plt.xlabel("Jumlah Produk per Transaksi")
plt.ylabel("Nilai Penjualan (satuan dataset)")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

korelasi = df["Quantity"].corr(df["Sales"])

if korelasi >= 0.5:
    hubungan = "positif cukup kuat atau kuat"
elif korelasi >= 0.2:
    hubungan = "positif lemah hingga sedang"
elif korelasi <= -0.5:
    hubungan = "negatif cukup kuat atau kuat"
elif korelasi <= -0.2:
    hubungan = "negatif lemah hingga sedang"
else:
    hubungan = "lemah atau hampir tidak ada hubungan linear"

print("\nINSIGHT 3 - SCATTER PLOT")
print(f"Korelasi Quantity dan Sales: {korelasi:.3f}.")
print(f"Hubungan linear kedua variabel tergolong {hubungan}. Periksa juga "
      "harga satuan dan diskon karena jumlah produk saja tidak menentukan "
      "nilai penjualan.")

In [ ]:
# ============================================================
# 7. VISUALISASI 4: HISTOGRAM (Distribusi Penjualan)
# ============================================================
plt.figure(figsize=(9, 6))
plt.hist(df["Sales"], bins=25, edgecolor="black")
plt.title("Distribusi Nilai Penjualan per Transaksi")
plt.xlabel("Nilai Penjualan (satuan dataset)")
plt.ylabel("Frekuensi Transaksi")
plt.tight_layout()
plt.show()

print("\nINSIGHT 4 - HISTOGRAM")
print(f"Rata-rata penjualan adalah {rata_transaksi:,.2f}, "
      f"sedangkan mediannya {median_penjualan:,.2f}.")

if rata_transaksi > median_penjualan:
    print("Rata-rata lebih tinggi daripada median. Hal ini dapat menunjukkan "
          "distribusi yang condong ke kanan akibat sejumlah transaksi bernilai besar.")
elif rata_transaksi < median_penjualan:
    print("Rata-rata lebih rendah daripada median. Periksa distribusi dan "
          "kemungkinan transaksi bernilai rendah yang memengaruhi rata-rata.")
else:
    print("Rata-rata dan median relatif sama. Periksa histogram untuk memastikan "
          "bentuk distribusi dan keberadaan nilai ekstrem.")

In [ ]:
# ============================================================
# 8. VISUALISASI 5: BOXPLOT (Deteksi Outlier dengan IQR)
# ============================================================
Q1_sales, Q3_sales, IQR_sales, batas_bawah, batas_atas = batas_iqr(df["Sales"])
outlier_sales = cari_outlier(df, "Sales")
outlier_profit = cari_outlier(df, "Profit")

plt.figure(figsize=(10, 5))
plt.boxplot(df["Sales"].dropna(), vert=False)
plt.title("Boxplot Nilai Penjualan dan Potensi Outlier")
plt.xlabel("Nilai Penjualan (satuan dataset)")
plt.tight_layout()
plt.show()

print("\nINSIGHT 5 - BOXPLOT DAN OUTLIER")
print(f"Q1 penjualan          : {Q1_sales:,.2f}")
print(f"Q3 penjualan          : {Q3_sales:,.2f}")
print(f"IQR penjualan         : {IQR_sales:,.2f}")
print(f"Batas bawah outlier   : {batas_bawah:,.2f}")
print(f"Batas atas outlier    : {batas_atas:,.2f}")
print(f"Jumlah potensi outlier: {len(outlier_sales)}")

if len(outlier_sales) > 0:
    print("Potensi outlier ditemukan. Nilai ekstrem bisa berasal dari pembelian "
          "dalam jumlah besar, produk berharga tinggi, atau kesalahan pencatatan. "
          "Data perlu diperiksa sebelum diputuskan untuk dihapus.")
    print("\nLima potensi outlier dengan penjualan terbesar:")
    kolom_tampil = [c for c in ["Order_ID", "Order_Date", "Category", "Product",
                                "Quantity", "Sales", "Profit"] if c in df.columns]
    display(outlier_sales.sort_values("Sales", ascending=False)[kolom_tampil].head())
else:
    print("Tidak ditemukan nilai penjualan di luar batas IQR. Ini tidak berarti "
          "data pasti bebas dari kesalahan.")

print("\nJumlah potensi outlier profit:", len(outlier_profit))

In [ ]:
# ============================================================
# 9. DASHBOARD RINGKAS (4 grafik dalam satu figure)
# ============================================================
judul("[9] DASHBOARD RINGKAS")

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("DASHBOARD ANALISIS PENJUALAN E-COMMERCE", fontsize=16, fontweight="bold")

# Penjualan per kategori
kategori_penjualan.sort_values().plot(kind="barh", ax=axes[0, 0], edgecolor="black")
axes[0, 0].set(title="Penjualan per Kategori", xlabel="Total Penjualan", ylabel="Kategori")

# Tren bulanan
axes[0, 1].plot(penjualan_bulanan.index, penjualan_bulanan.values, marker="o")
axes[0, 1].set(title="Tren Penjualan Bulanan", xlabel="Bulan", ylabel="Total Penjualan")
axes[0, 1].tick_params(axis="x", rotation=45)
axes[0, 1].grid(True, alpha=0.3)

# Distribusi penjualan
axes[1, 0].hist(df["Sales"], bins=25, edgecolor="black")
axes[1, 0].set(title="Distribusi Penjualan", xlabel="Nilai Penjualan", ylabel="Frekuensi")

# Quantity vs Sales
axes[1, 1].scatter(df["Quantity"], df["Sales"], alpha=0.5)
axes[1, 1].set(title="Quantity vs Sales", xlabel="Jumlah Produk", ylabel="Nilai Penjualan")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

print("Dashboard berhasil dibuat dengan 4 grafik.")

In [ ]:
# ============================================================
# 10. TEMUAN UTAMA
# ============================================================
judul("[10] TEMUAN UTAMA DAN REKOMENDASI")


def ringkasan_per(kolom):
    """Total sales, profit, dan jumlah transaksi per kelompok."""
    return (
        df.groupby(kolom)
          .agg(Total_Sales=("Sales", "sum"),
               Total_Profit=("Profit", "sum"),
               Jumlah_Transaksi=("Sales", "size"))
          .sort_values("Total_Sales", ascending=False)
          .round(2)
    )


print("\nPerforma kategori:")
display(ringkasan_per("Category"))

print("\nPerforma kota:")
display(ringkasan_per("City"))

print("\nTEMUAN UTAMA")
print(f"1. Kategori penjualan tertinggi: {kategori_terlaris}")
print(f"2. Kategori profit tertinggi   : {kategori_profit_tertinggi}")
print(f"3. Kota penjualan tertinggi    : {wilayah_tertinggi}")
print(f"4. Kota penjualan terendah     : {kota_terendah}")
print(f"5. Rata-rata rating pelanggan  : {rata_rating:.2f}/5")
print(f"6. Potensi outlier penjualan   : {len(outlier_sales)}")
print(f"7. Potensi outlier profit      : {len(outlier_profit)}")

In [ ]:
# ============================================================
# 11. LAPORAN REKOMENDASI BISNIS (target 200-300 kata)
# ============================================================
judul("[11] LAPORAN REKOMENDASI BISNIS")

# --- Kalimat rekomendasi yang menyesuaikan hasil data ---
if rata_rating < 3:
    rekomendasi_rating = (
        "Rating pelanggan tergolong rendah, sehingga kualitas produk, "
        "layanan, dan pengiriman perlu dievaluasi dengan mengelompokkan "
        "keluhan pelanggan.")
elif rata_rating < 4:
    rekomendasi_rating = (
        "Rating pelanggan masih dapat ditingkatkan melalui peninjauan "
        "ulasan, ketepatan pengiriman, dan kualitas produk.")
else:
    rekomendasi_rating = (
        "Rating pelanggan relatif baik, sehingga kualitas produk dan "
        "layanan perlu dijaga.")

if total_profit > 0:
    rekomendasi_profit = (
        "Profit agregat positif, namun profit per kategori, diskon, dan "
        "tingkat pengembalian tetap perlu dibandingkan sebelum menambah "
        "anggaran promosi.")
elif total_profit < 0:
    rekomendasi_profit = (
        "Profit agregat negatif, sehingga struktur biaya, diskon, harga "
        "jual, dan kategori yang merugi perlu diperiksa sebelum promosi "
        "diperluas.")
else:
    rekomendasi_profit = (
        "Profit agregat nol, sehingga pencatatan keuangan dan struktur "
        "biaya perlu diperiksa terlebih dahulu.")

if rata_transaksi > median_penjualan:
    rekomendasi_distribusi = (
        "Rata-rata transaksi melebihi median, sehingga keputusan sebaiknya "
        "mempertimbangkan median dan distribusi, bukan rata-rata saja.")
else:
    rekomendasi_distribusi = (
        "Rata-rata transaksi tidak melebihi median, sehingga distribusi "
        "nilai transaksi dan segmen pelanggan dapat dipelajari lebih lanjut.")

laporan = f"""LAPORAN ANALISIS DATA PENJUALAN E-COMMERCE

Analisis mencakup {len(df):,} baris transaksi setelah pembersihan. Sebanyak {duplikat_awal} baris duplikat dihapus, data numerik kosong diisi median, dan data kategorikal kosong diisi modus atau Unknown. Mata uang belum dapat dipastikan, sehingga nilai disajikan dalam satuan asli dataset.

Total penjualan tercatat {total_penjualan:,.2f}, dengan rata-rata transaksi {rata_transaksi:,.2f} dan median {median_penjualan:,.2f}. Total profit mencapai {total_profit:,.2f}. Kategori penjualan tertinggi adalah {kategori_terlaris}, sedangkan profit tertinggi ada pada {kategori_profit_tertinggi}. Kota dengan penjualan tertinggi adalah {wilayah_tertinggi} dan terendah adalah {kota_terendah}. Korelasi Quantity dan Sales sebesar {korelasi:.3f} tidak membuktikan sebab-akibat. Metode IQR menemukan {len(outlier_sales)} potensi outlier penjualan yang perlu diperiksa, bukan langsung dihapus.

Rekomendasi utama: jaga ketersediaan stok kategori berpenjualan tinggi, evaluasi kategori berpenjualan rendah menggunakan data profit, dan uji promosi secara terukur di kota yang potensial. {rekomendasi_rating} {rekomendasi_profit} {rekomendasi_distribusi}

Kesimpulannya, keputusan bisnis perlu didasarkan pada kombinasi penjualan, profit, tren waktu, dan kualitas data. Analisis lanjutan terhadap diskon, retur, dan biaya operasional akan membantu menguji rekomendasi ini.
"""

jumlah_kata = len(laporan.split())

print(laporan)
print("-" * LEBAR)
print("Jumlah kata laporan:", jumlah_kata)

if 200 <= jumlah_kata <= 300:
    print("STATUS: Laporan memenuhi ketentuan 200-300 kata.")
else:
    print("CATATAN: Jumlah kata di luar 200-300. Sesuaikan isi laporan.")

with open("laporan_rekomendasi.txt", "w", encoding="utf-8") as f:
    f.write(laporan)

print("\nFile laporan berhasil dibuat: laporan_rekomendasi.txt")

In [ ]:
# ============================================================
# 12. UNDUH HASIL
# ============================================================
judul("ANALISIS SELESAI")
print("File yang diunduh:")
print("1. ecommerce_sales_cleaned.csv")
print("2. laporan_rekomendasi.txt")

files.download("ecommerce_sales_cleaned.csv")
files.download("laporan_rekomendasi.txt")

## 12. Kesimpulan
1. Kategori Fashion dan Electronics adalah penyumbang penjualan terbesar, sehingga stoknya perlu dijaga dan di-restock lebih dulu.
2. Dhaka adalah pasar utama; kota lain seperti Sylhet dan Chattogram dapat dikembangkan lewat promosi yang diuji secara terukur.
3. Batasi diskon besar (di atas 30%) karena profit per transaksi turun drastis.
4. Selidiki penurunan penjualan Oktober 2025 dan tren 2025 yang lebih rendah dari 2024.
5. Tingkatkan rating (3,96) lewat perbaikan pengiriman dan pengemasan, serta tinjau penyebab retur dan transaksi rugi.

**Keterbatasan:** data kosong pada kategori dan kota diisi dengan modus, sehingga Fashion dan Dhaka dapat terlihat sedikit lebih besar dari kondisi sebenarnya. Mata uang dataset belum dipastikan, sehingga nilai disajikan dalam satuan asli dataset.